In [0]:
PATH_PRODUCT_FILES = "/Volumes/customer_360/raw/source_files/landing_data/products/"

PATH_PRODUCT_CHECKPOINTLOCATION_BRONZE = (
    "/Volumes/customer_360/raw/source_files/checkpoints/products/"
)

TABLE_BRONZE_PRODUCT = "customer_360.bronze.products"

TABLE_METRIC = "customer_360.raw.stream_metrics"

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS customer_360.bronze.products (

    product_id STRING NOT NULL,
    product_name STRING,
    product_category STRING,
    product_subcategory STRING,
    product_price DECIMAL(12,2),
    product_status STRING,
    updated_at TIMESTAMP

)
USING DELTA
""")

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DecimalType,
    TimestampType
)

product_schema = StructType([

    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), True),
    StructField("product_category", StringType(), True),
    StructField("product_subcategory", StringType(), True),
    StructField("product_price", DecimalType(12,2), True),
    StructField("product_status", StringType(), True),
    StructField("updated_at", TimestampType(), True)

])

In [0]:
product_bronze = (
    spark
    .readStream
    .format("csv")
    .option("header", True)
    .schema(product_schema)
    .load(PATH_PRODUCT_FILES)
)

In [0]:
query = (
    product_bronze
    .writeStream
    .trigger(availableNow=True)
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        PATH_PRODUCT_CHECKPOINTLOCATION_BRONZE
    )
    .toTable(TABLE_BRONZE_PRODUCT)
)

query.awaitTermination()

In [0]:
import json
from pyspark.sql import Row
from datetime import datetime

metrics = []

for p in query.recentProgress:

    progress = json.loads(p.json)

    source = progress["sources"][0]

    metrics.append(
        Row(
            metric_time=datetime.now(),
            query_name="product_bronze",
            batch_id=int(progress["batchId"]),
            input_rows=int(source.get("numInputRows", 0)),
            input_rows_per_second=float(
                source.get("inputRowsPerSecond", 0.0)
            ),
            processed_rows_per_second=float(
                source.get("processedRowsPerSecond", 0.0)
            ),
            processing_time_ms=int(
                progress.get("durationMs", {})
                .get("triggerExecution", 0)
            )
        )
    )

if metrics:

    metrics_df = spark.createDataFrame(metrics)

    metrics_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(TABLE_METRIC)

In [0]:
display(
    spark.sql(f"select * from {TABLE_BRONZE_PRODUCT}")
)